# Retention Probability Calibration

Checkpoint 42 compares uncalibrated, sigmoid, and isotonic probabilities for the provisional Logistic Regression. Calibration mappings learn from out-of-fold 2023 predictions, evaluation uses 2024, and the 2025 test target remains locked.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PROCESSED = PROJECT_ROOT / 'data' / 'processed'

metrics = pd.read_csv(PROCESSED / 'retention_calibration_metrics.csv')
reliability = pd.read_csv(PROCESSED / 'retention_reliability_bins.csv')
differences = pd.read_csv(PROCESSED / 'retention_calibration_differences.csv')
selection = pd.read_csv(PROCESSED / 'retention_calibration_selection.csv')
validation = pd.read_csv(PROCESSED / 'retention_calibration_validation.csv')

print(f'Calibration methods: {len(metrics):,}')
print(f'Reliability bins: {len(reliability):,}')
print(f"Checks passed: {(validation['status'] == 'PASS').sum()}/{len(validation)}")

## Probability-quality metrics

Lower Brier score, log loss, and expected calibration error are better. PR-AUC and ROC-AUC describe ranking rather than calibration.

In [ ]:
metrics[[
    'method',
    'brier_score',
    'log_loss',
    'expected_calibration_error',
    'mean_predicted_probability',
    'observed_attrition_rate',
    'pr_auc',
    'roc_auc',
    'selected_method',
]].round(4)

In [ ]:
figure, axis = plt.subplots(figsize=(8, 4.5))
axis.bar(metrics['method'], metrics['brier_score'], color=['#8c8c8c', '#2f6b9a', '#70a37f'])
axis.set_title('Validation Brier Score by Calibration Method')
axis.set_ylabel('Brier score — lower is better')
axis.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Reliability diagram

A calibrated method should remain close to the diagonal, where predicted probability equals observed attrition rate.

In [ ]:
figure, axis = plt.subplots(figsize=(7, 6))
for method, group in reliability.groupby('method', sort=False):
    axis.plot(
        group['mean_predicted_probability'],
        group['observed_attrition_rate'],
        marker='o',
        label=method,
    )
axis.plot([0, 1], [0, 1], linestyle='--', color='black', label='Perfect calibration')
axis.set_xlim(0, 0.65)
axis.set_ylim(0, 0.30)
axis.set_xlabel('Mean predicted probability')
axis.set_ylabel('Observed attrition rate')
axis.set_title('Validation Reliability Diagram')
axis.legend()
axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Paired uncertainty

A positive Brier difference for method A minus method B means method B has the lower, better score.

In [ ]:
differences[[
    'method_a',
    'method_b',
    'observed_brier_difference_a_minus_b',
    'difference_lower_95',
    'difference_upper_95',
    'conclusion',
]].round(4)

## Selection decision

In [ ]:
selection[[
    'method',
    'validation_brier_score',
    'brier_improvement_vs_uncalibrated',
    'eligible_for_selection',
    'selected_method',
    'recommended_dashboard_label',
]].round(4)

Sigmoid calibration is selected because it materially improves probability accuracy, is statistically indistinguishable from isotonic calibration on Brier score, preserves ranking, and is the simpler smooth mapping. The result remains validation-stage evidence from synthetic data.